In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.functions import date_format
from pyspark.sql import types as t
from pyspark.sql.window import Window
from datetime import datetime
import logging        
from config import ROUTES, PipelineConfig  

In [0]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

In [0]:
# Config 
GOLD_FATO_PATH   = "workspace.case_spark_cvm.gold_fato_diario"
NOME_TABELA      = f"gold_cubo_comparativo" 
GOLD_PATH        = f"{ROUTES.TABLE_BASE}.{NOME_TABELA}"
DATA_PROC        = int(datetime.now().strftime("%Y%m%d"))

### 1. gold_fato_diario

In [0]:
gold_fato_diario  = spark.read.table(GOLD_FATO_PATH)

In [0]:
df_teste = gold_fato_diario.filter(f.col("cnpj_fundo_classe") == "41240381000163") \
  .select("dt_comptc", "vl_quota", "retorno_252d") \
  .orderBy("dt_comptc")

In [0]:
display(df_teste)

### 3. Selecionando de Comparativo e Filtrando pela Ultima Data

In [0]:
#Encontrando a Ultima Data do Fundo
window_ultima_data = Window.partitionBy("cnpj_fundo_classe").orderBy(f.col("dt_comptc").desc())

window_retorno = Window.orderBy(f.col("retorno_252d").desc_nulls_last())
window_captacao = Window.orderBy(f.col("captacao_liquida_252d").desc_nulls_last())
window_sharpe = Window.orderBy(f.col("sharpe_252d").desc_nulls_last())
window_vol = Window.orderBy(f.col("volatilidade_252d").asc_nulls_last())

gold_cubo_comparativo = gold_fato_diario\
    .withColumn("rn", f.row_number().over(window_ultima_data))\
    .filter(f.col("rn") == 1)\
    .select(
        "cnpj_fundo_classe",
        f.col("dt_comptc").alias("dt_referencia"),
        f.col("vl_patrim_liq").alias("pl_atual"),
        "retorno_21d",
        "retorno_252d",
        "captacao_liquida_252d",
        "sharpe_252d",
        "volatilidade_252d",
        "nr_cotst"
    )\
    .coalesce(1) 


gold_cubo_comparativo = gold_cubo_comparativo \
    .withColumn("rank_retorno_1a", f.rank().over(window_retorno))\
    .withColumn("rank_captacao_1a", f.rank().over(window_captacao))\
    .withColumn("rank_sharpe_1a", f.rank().over(window_sharpe))\
    .withColumn("rank_volatilidade", f.rank().over(window_vol))

In [0]:
gold_cubo_comparativo.display()

### 4. Salvando os dados

In [0]:

log.info(f"Iniciando a escrita da dimensão unificada em: {GOLD_PATH}") 

PipelineConfig.gravar_cubo_gold(
    spark=spark,
    df_cubo=gold_cubo_comparativo,
    tabela_destino=GOLD_PATH,
    zorder_cols=["cnpj_fundo_classe" ,"rank_retorno_1a"]
)


log.info(f"Processamento da {GOLD_PATH} concluído com sucesso!")